<span style="color:red;font-size:2em;font-weight:bold"> PARTIE 2 - Tracking des expérimentations via MLFlow</span>

<span style="color:blue;font-size:1.5em;font-weight:bold;background-color:yellow"> Modules </span>

In [ ]:
# Module pour recharger un module sans redemarrer le kernel
# import importlib
%load_ext autoreload
%autoreload 2

In [ ]:
# Roots
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

# Stats
from scipy.stats import zscore, chi2_contingency, f_oneway, chi2

#Selection
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV, 
    cross_validate,
    StratifiedShuffleSplit
)
# Metrics
from sklearn.metrics import (
    accuracy_score, auc, classification_report, confusion_matrix, f1_score, fbeta_score, precision_recall_curve, 
    precision_score, recall_score, roc_auc_score, roc_curve, ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay
)

# Feature importance
from sklearn.inspection import permutation_importance

#Preprocess
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer, RobustScaler,PowerTransformer

#Modèles
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from catboost import CatBoostClassifier, Pool

#SHAP
import shap

In [ ]:
# Sert à éviter les Warnings avec les transformations sur des vues en transformant 
# ces warning en erreur obligeant ainsi à ne travailler que sur des copies ou les originaux.

pd.set_option('mode.chained_assignment','raise')

In [ ]:
# Ajoute le dossier datas_manipulation au sys.path. Remarque ne pas oublier le __init__.py dans le dossier datas_manipulation
import sys
# root_path = Path(__file__).resolve().parents[1] # Ne fonctionne pas sur notebook
root_path = Path.cwd().parent
sys.path.append(str(root_path))

In [ ]:
# Fonctions personnelles
from notebooks.datas_manipulation.quick_clean_datas import (
    drop_empty_columns, clean_infinites, drop_col_with_unique_value
)
from notebooks.datas_manipulation.datas_assembler import merging_data
from notebooks.datas_manipulation.memory_optimizer import optimize_dtypes, float_to_int
from notebooks.datas_manipulation.export_datas import export_datas

from notebooks.utils.feature_aggregator import agg_features, agg_columns


# A remodeler
# Ajoute le dossier utils au sys.path. 
# Remarque ne pas oublier le __init__.py dans le dossier utils
import sys
utils_path = Path.cwd().parent/"utils"
sys.path.append(str(utils_path))
# Fonctions personnelles
# Suivi des filtrage des features et observations
from filtering_counters import cleaning_counter, cleaning_results, removedAndAdded_col 
#, graph_hyperParamEffect, graph_importance # Fonctions de plotting
from plotting import create_fig, make_figure, graphs
# Fonctions de correlation
from correlation import graphs_corr, CorrCouples_VIF, chi2_test, anova_test 
# Divers fonctions
from misc import features_types, data_props, top_score, iqr_outliers,zmad_outliers 

In [ ]:
# Paramètres globaux

# Création dossier results
save_path = root_path.joinpath('datas/results')
Path.mkdir(save_path,exist_ok = True)

<span style="color:blue;font-size:1.5em;font-weight:bold;background-color:yellow"> Datasets </span>

In [ ]:
# Chemin du dataset d'entrainement/test du modèle
datas_path = (
    root_path /'datas'/'raw_datas'/
    'Projet+Mise+en+prod+-+home-credit-default-risk'/'finale_datasets'
)